## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-03-18-58-29 +0000,nypost,LA woman killed by LAPD patrol car responding ...,https://nypost.com/2026/11/03/us-news/woman-ki...
1,2026-11-03-18-52-09 +0000,nypost,4-year-old boy’s lifeless body found buried un...,https://nypost.com/2026/11/03/us-news/4-year-o...
2,2026-11-03-18-50-47 +0000,nypost,NYC Marathon runner Rachel Apa was lifelong at...,https://nypost.com/2026/11/03/us-news/nyc-mara...
3,2026-11-03-18-47-45 +0000,nyt,Here’s the latest.,https://www.nytimes.com/live/2026/11/03/us/mid...
4,2026-11-03-18-43-45 +0000,nypost,Escaped Tennessee inmate convicted of co-worke...,https://nypost.com/2026/11/03/us-news/escaped-...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2915/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
135,trump,42
50,election,39
51,day,28
116,midterm,21
262,voters,17


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
389,2026-11-02-19-34-04 +0000,chitri,President Donald Trump spends day before midte...,https://www.chicagotribune.com/2026/11/02/dona...,186
213,2026-11-03-10-56-00 +0000,wsj,Trump surprises Republicans with an election-e...,https://www.wsj.com/politics/elections/control...,174
275,2026-11-03-02-55-07 +0000,nypost,Trump says Republicans should ‘win in a landsl...,https://nypost.com/2026/11/02/us-news/trump-sa...,140
19,2026-11-03-18-11-21 +0000,nypost,Alaska Senate candidate ‘Decoy’ Dan Sullivan m...,https://nypost.com/2026/11/03/us-news/alaska-s...,138
292,2026-11-03-00-43-43 +0000,nypost,"Xavier Becerra, Steve Hilton make final push a...",https://nypost.com/2026/11/02/us-news/xavier-b...,134


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
389,186,2026-11-02-19-34-04 +0000,chitri,President Donald Trump spends day before midte...,https://www.chicagotribune.com/2026/11/02/dona...
329,101,2026-11-02-22-40-00 +0000,wsj,Midterm candidates across the country are maki...,https://www.wsj.com/politics/policy/midterm-ca...
286,81,2026-11-03-01-18-57 +0000,nypost,8 California House races to watch: GOP vs. GOP...,https://nypost.com/2026/11/02/us-news/8-califo...
281,81,2026-11-03-02-00-00 +0000,wsj,Tuesday’s midterm elections will offer clues t...,https://www.wsj.com/politics/elections/what-th...
213,57,2026-11-03-10-56-00 +0000,wsj,Trump surprises Republicans with an election-e...,https://www.wsj.com/politics/elections/control...
87,49,2026-11-03-16-12-00 +0000,wsj,Trump Says U.S. Plans to Cut Off Military Aid ...,https://www.wsj.com/world/middle-east/trump-sa...
204,47,2026-11-03-11-00-00 +0000,nypost,Campaign to keep Ilhan Omar’s secrets safe inc...,https://nypost.com/2026/11/03/us-news/campaign...
176,45,2026-11-03-11-57-00 +0000,wsj,"Follow the WSJ’s Election Day live coverage, a...",https://www.wsj.com/politics/elections/your-ho...
233,41,2026-11-03-10-00-00 +0000,wsj,"Election Day is here, with control of the Hous...",https://www.wsj.com/articles/midterms-2026-ele...
189,38,2026-11-03-11-18-56 +0000,cbc,"Dozens still missing in Russian, guided-bomb a...",https://www.cbc.ca/news/world/ukraine-russia-m...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
